# K-Means clustering on `combined_cleaned.csv`

This notebook loads `combined_cleaned.csv`, prepares features, runs **K-Means clustering**, visualizes clusters, and saves `combined_cleaned_kmeans.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

%matplotlib inline

In [ ]:
# STEP 2: Load dataset
data = pd.read_csv('combined_cleaned.csv')
print('Data shape:', data.shape)
display(data.head())

In [ ]:
# STEP 3: Build feature matrix (exclude target if present)
if 'heart_disease' in data.columns:
    X = data.drop(columns=['heart_disease'])
else:
    X = data.copy()

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)
print('Encoded features shape:', X.shape)

In [ ]:
# STEP 4: Impute + scale (recommended for K-Means)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Clustering matrix shape:', X_scaled.shape)

In [ ]:
# STEP 5: Choose k (Elbow + Silhouette) - silhouette sampled for speed
k_values = range(2, 11)
inertias = []
sil_scores = []

silhouette_sample_size = min(10000, X_scaled.shape[0])

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(
        silhouette_score(
            X_scaled,
            labels,
            sample_size=silhouette_sample_size,
            random_state=42,
        )
    )

best_k = int(list(k_values)[int(np.argmax(sil_scores))])
print('Best k by silhouette:', best_k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set_title('Elbow method (inertia)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_values), sil_scores, marker='o')
axes[1].set_title(f'Silhouette score (sample={silhouette_sample_size})')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 6: Fit final K-Means, visualize (PCA), and save labels
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X_scaled)

out = data.copy()
out['cluster'] = clusters

print('Cluster counts:')
print(out['cluster'].value_counts().sort_index())

# PCA 2D plot
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, s=5, cmap='tab10', alpha=0.6)
plt.title(f'K-Means clusters (k={best_k}) visualized with PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

out_path = 'combined_cleaned_kmeans.csv'
out.to_csv(out_path, index=False)
print('Saved:', out_path)

# Agglomerative clustering (sampled)

Agglomerative clustering builds a hierarchical structure and is typically expensive for large datasets (it can become slow due to pairwise distance work). To keep runtime reasonable, this notebook fits Agglomerative clustering on a random sample of rows, then assigns cluster labels to the full dataset using nearest-centroid approximation.

For visualization, it uses scatter plots with `bmi`, `glucose`, and `age`.

In [ ]:
# STEP 7: Agglomerative Clustering (sampled)
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances_argmin_min

sample_size = min(10000, X_scaled.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_scaled.shape[0], size=sample_size, replace=False)

X_sample = X_scaled[sample_idx]

# Using the same `best_k` selected by silhouette for K-Means
agg = AgglomerativeClustering(n_clusters=best_k, linkage='average')
labels_sample = agg.fit_predict(X_sample)

# Build centroids from the sampled data and assign labels to all rows.
# This is an approximation because Agglomerative doesn't provide a native `predict`.
centroids = np.vstack([
    X_sample[labels_sample == i].mean(axis=0)
    for i in range(best_k)
])

clusters_full, _ = pairwise_distances_argmin_min(X_scaled, centroids, metric='euclidean')

out2 = data.copy()
out2['agglomerative_cluster'] = clusters_full

print('Agglomerative cluster counts:')
print(out2['agglomerative_cluster'].value_counts().sort_index())

# Visualization using BMI / glucose / age if those columns exist
required_cols = ['bmi', 'glucose', 'age']
if all(c in out2.columns for c in required_cols):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(out2['bmi'], out2['glucose'], c=out2['agglomerative_cluster'], s=5, cmap='tab10', alpha=0.6)
    axes[0].set_title('BMI vs Glucose')
    axes[0].set_xlabel('BMI')
    axes[0].set_ylabel('Glucose')

    axes[1].scatter(out2['bmi'], out2['age'], c=out2['agglomerative_cluster'], s=5, cmap='tab10', alpha=0.6)
    axes[1].set_title('BMI vs Age')
    axes[1].set_xlabel('BMI')
    axes[1].set_ylabel('Age')

    axes[2].scatter(out2['glucose'], out2['age'], c=out2['agglomerative_cluster'], s=5, cmap='tab10', alpha=0.6)
    axes[2].set_title('Glucose vs Age')
    axes[2].set_xlabel('Glucose')
    axes[2].set_ylabel('Age')

    plt.tight_layout()
    plt.show()
else:
    print('Columns bmi/glucose/age not found; skipping BMI/glucose/age scatter visualization.')

out_path2 = 'combined_cleaned_agglomerative.csv'
out2.to_csv(out_path2, index=False)
print('Saved:', out_path2)